# Атака на ошибки при работе RSA CRT (КТО)
Поскольку у RSA большое энергопотребление и он требует большой вычислительной мощности, на слабых устройствах часто используется вариант расшифрования на основе Китайской Теоремы об Остатках.
Предварительно вычисляются значения $dP=e^{-1}\space mod \space (p-1)$, $dQ=e^{-1}\space mod \space (q-1)$ и $qInv=q^{-1}\space mod \space p$.
После этого алгоритм расшифрования работает следующим образом:
$$M_p=C^{dP}\space mod \space p$$
$$M_q=C^{dQ}\space mod \space q$$
$$h=qInv\cdot(M_p-M_q)\space mod\space p$$
$$M=M_q+h\cdot q$$
$M_p=M\space mod \space p$ и $M_q=M\space mod\space q$

Алгоритм отлично работает, сильно уменьшая вычислительную сложность RSA, но у него есть один недостаток. Если внедрить ошибку(фолт) при вычислении одного из остатков $M$, то получится $M'$, для которого:
$$M'_p=M_p$$
$$M'_q\ne M_q$$
Раз $M'_p=M_p$, то $M-M'=kp$, для некоторого $k \in \mathbb{Z}$, и $GCD(M-M',N)=p$, что позволяет факторизовать $N$ и вычислить $d$.

Получите с сервера $M$, $M'$, вычислите $d$ и отправьте на сервер, чтобы получить флаг. Удачи!

In [1]:
import socket
import re
from Crypto.Util.number import bytes_to_long, long_to_bytes, inverse, GCD
class VulnServerClient:
    def __init__(self,show=True):
        """Инициализация, подключаемся к серверу"""
        self.s=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
        self.s.connect(('cryptotraining.zone',1340))
        if show:
            print (self.recv_until().decode())
    def recv_until(self,symb=b'\n>'):
        """Получаем сообщения с сервера, по дефолту до знака приглашения"""
        data=b''
        while True:
            
            data+=self.s.recv(1)
            if data[-len(symb):]==symb:
                break
        return data
    def get_public_key(self,show=True):
        """Получаем открытый ключ с сервера"""
        self.s.sendall('public\n'.encode())
        response=self.recv_until().decode()
        if show:
            print (response)
        e=int(re.search('(?<=e: )\d+',response).group(0))
        N=int(re.search('(?<=N: )\d+',response).group(0))
        self.num_len=len(long_to_bytes(N))
        return (e,N)
    
    def decryptBytes(self,m,show=True):
        """Получить открытый текст от выбранного шифротекста в байтах с сервера"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        if len(m)>num_len:
            print ("The message is too long")
            return
        if len(m)<num_len:
            m=bytes((num_len-len(m))*[0x0])+m
        hex_m=m.hex().encode()
        self.s.sendall(b'decrypt '+hex_m+b'\n')
        response=self.recv_until().decode()
        if show:
            print (response)
        if response.find('flag')!=-1:
            print('You tried to submit \'flag\'')
            return None
        signature_hex=re.search('(?<=Signature: )[0-9a-f]+',response).group(0)
        signature_bytes=bytes.fromhex(signature_hex)
        return bytes_to_long(signature_bytes)
    
    
    def decryptNumber(self,m,show=True):
        """Получить открытый текст для выбранного закрытого текста в виде числа с сервера"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        return self.decryptBytes(long_to_bytes(m,num_len),show)
    
    def faultyDecryptBytes(self,m,show=True):
        """Получить открытый текст с ошибкой в алгоритме для выбранного закрытого текста в байтах с сервера"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        if len(m)>num_len:
            print ("The message is too long")
            return
        if len(m)<num_len:
            m=bytes((num_len-len(m))*[0x0])+m
        hex_m=m.hex().encode()
        self.s.sendall(b'faulty_decrypt '+hex_m+b'\n')
        response=self.recv_until().decode()
        if show:
            print (response)
        if response.find('flag')!=-1:
            print('You tried to submit \'flag\'')
            return None
        signature_hex=re.search('(?<=Signature: )[0-9a-f]+',response).group(0)
        signature_bytes=bytes.fromhex(signature_hex)
        return bytes_to_long(signature_bytes)
    
    
    def faultyDecryptNumber(self,m,show=True):
        """Получить открытый текст с ошибкой в алгоритме для выбранного закрытого текста в числовом представлении с сервера"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        return self.faultyDecryptBytes(long_to_bytes(m,num_len),show)
        
    def checkDNumber(self,c,show=True):
        """Проверить, является ли это число d"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        signature_bytes=long_to_bytes(c,num_len)
        self.checkDBytes(signature_bytes,show)
    
    def checkDBytes(self,c,show=True):
        """Проверить, является ли эта последовательность байтов d"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        if len(c)>num_len:
            print ("The message is too long")
            return
        
        hex_c=c.hex().encode()
        self.s.sendall(b'flag '+hex_c+b'\n',)
        response=self.recv_until(b'\n').decode()
        
        if show:
            print (response)
        
        if response.find('Wrong')!=-1:
            print('Wrong signature')
            x=self.recv_until()
            if show:
                print (x)
            return
        flag=re.search('CRYPTOTRAINING\{.*\}',response).group(0)
        print ('FLAG: ',flag)
        
    def __del__(self):
        self.s.close()


<>:26: SyntaxWarning: invalid escape sequence '\d'
<>:27: SyntaxWarning: invalid escape sequence '\d'
<>:133: SyntaxWarning: invalid escape sequence '\{'
<>:26: SyntaxWarning: invalid escape sequence '\d'
<>:27: SyntaxWarning: invalid escape sequence '\d'
<>:133: SyntaxWarning: invalid escape sequence '\{'
/var/folders/jm/fq05587x6d787nhsnkqr2zy40000gp/T/ipykernel_89663/3103748132.py:26: SyntaxWarning: invalid escape sequence '\d'
  e=int(re.search('(?<=e: )\d+',response).group(0))
/var/folders/jm/fq05587x6d787nhsnkqr2zy40000gp/T/ipykernel_89663/3103748132.py:27: SyntaxWarning: invalid escape sequence '\d'
  N=int(re.search('(?<=N: )\d+',response).group(0))
/var/folders/jm/fq05587x6d787nhsnkqr2zy40000gp/T/ipykernel_89663/3103748132.py:133: SyntaxWarning: invalid escape sequence '\{'
  flag=re.search('CRYPTOTRAINING\{.*\}',response).group(0)


In [43]:
vs=VulnServerClient()
(e,N)=vs.get_public_key()

Welcome to RSA CRT Decryption Faults task
Available commands:
help - print this help
public - show public key
decrypt <hex(data)> - decrypt ciphertext
faulty_decrypt <hex(data)> - decrypt with fault
flag <hex(d))> - print flag 
quit - quit
>
e: 65537
N: 20159717663186764200842482638329142432479376755681286432561400011207751568770239378735042390550988864636478212097889382541806378632813451522011734778394352464750695430236459156439656932108536936107092785759187120915559173321302027525229018106368725032056109022369913503577023942696069608771010384365856481001383579432844112231215767630328627015097422540087789462404508697086321213990868031273219614897901436844999442259387453021270642395531884848697650933478124254071912232445708062597679170291021925633789812405697682134528381868778865376836541179591638312152472136313757252384761293684336082840137773984575947459061
>


Получаем сообщения с сервера для некоторого шифртекста:

In [44]:
import random

random_ciphertext = random.randrange(2, N)

message_correct = vs.decryptNumber(random_ciphertext)
message_with_error = vs.faultyDecryptNumber(random_ciphertext)

Signature: 9be4d04a0735001a749a150b6b7c911a9cfc3a3e4ec5e235977b540204d1c56b745810b94cba79eb34d276c4b98bc4a548871607d73c16a6de266ffccb9a429e207f278c4ff88aba837ff19945dd1804356641f4e55185c5bec9b1b38a57579ea45acbf62d53de37f93027449092329dc26f7932b0886eeeca925c83886eff683bcaa544bd7f1ec71ae7cfb336850c53122155032f51977fd1fcd385579e7b80852836fa75fc09abafa8cea6b33b7afa0e968949de562b642bfae3e59811832f1460a8cb0966cb597001ca32b45749afc1d7b104ca382a37a67064f317e0e82b08bed02961f6c16cc5492b2bc3fc83d9286da293a33f9ecc6a262929a7c41704
>
Signature: 537b7ad9896a8006811a5a799ff15d7677717313a5f5c46339f8cfc7fb5add23258d7ac9d937c1e87d31236f870f3d532bfda2c40a24c1b543ca8ab30b3d30aef04881f9149587755bbcafe26fc2a2e3cf6cc0f8eb3bad17e0fee40cf962ca4d0f1f1a814149afead2480e319351277685156c19f1fc6aa453399c958a559b0c7667849c4fbe781ae30f2ac4bf099169e4f177b8f22b794fff3e73801be772a51ab9e49168ad2a04be1607124a0df8bd7be65ee0e08dbd70ebd12e02545ff431d399a866abce968a0edd92d1b1461cde8504541361bacab1a6cff9fd1277e60fb282fce8a70111d

In [45]:
message_with_error.bit_length(), message_correct.bit_length()

(2039, 2048)

Получаем ```d```

In [50]:
p = GCD(abs(message_correct - message_with_error), N)
q = N // p

phi = (p - 1)*(q - 1)
d = inverse(e, phi)

print(d)

7475801735941955389674151327636497681257095889243058183477115285600225618747015084935192871470477476498166379123472323480072637134950719629208709436185939972089600546882473516928650722813582757193809237107367206305000756660018053541444387552666114171446047539110060086324250619945108560079983297545560676530352673196882100867409466823519383404044484958913516776131481967490792058563417783293136082659392766788990413545041962418779312091691640838955546332083282091261957514329004087004440691438595342004784399790790014734523090280464011586344252007320919729564253113548064027521028822186124920824856004524445404345941


Отправляем на сервер:

In [51]:
vs.checkDBytes(long_to_bytes(d))

Congratulations! Here is your flag: CRYPTOTRAINING{f41ty_cr7_1s_cl34r_4nd_pr353n7_d4ng3r}

FLAG:  CRYPTOTRAINING{f41ty_cr7_1s_cl34r_4nd_pr353n7_d4ng3r}
